# 03 - 预测未见组合扰动

使用训练好的 GEARS 模型预测未见单基因和双基因组合扰动，计算评估指标。

In [ ]:
import sys
sys.path.insert(0, '..')
from gears import PertData, GEARS
import torch
import numpy as np
import pandas as pd

# 加载数据和模型
pert_data = PertData('../data')
pert_data.load(data_name='norman')
pert_data.prepare_split(split='simulation', seed=1)
pert_data.get_dataloader(batch_size=32, test_batch_size=128)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = GEARS(pert_data, device=device)
model.model_initialize(hidden_size=64)

# 加载已训练模型 (如果存在)
model_path = '../models/gears_norman_hidden64_epoch20.pt'
import os
if os.path.exists(model_path):
    model.model.load_state_dict(torch.load(model_path, map_location=device))
    print('Loaded pre-trained model')
else:
    print('No pre-trained model found. Train first with 02_train_gears_norman.ipynb')

In [ ]:
# 预测未见组合扰动
predictions = model.predict([
    ['CBL', 'CNN1'],
    ['FEV']
])

print('Predictions:')
for key, val in predictions.items():
    print(f'  {key}: shape={val.shape}')

In [ ]:
# 在测试集上评估
# GEARS 内置评估函数
if hasattr(model, 'evaluate'):
    metrics = model.evaluate()
    print('Evaluation metrics:')
    for k, v in metrics.items():
        print(f'  {k}: {v}')
else:
    print('Manual evaluation:')
    print('  - Top-20 DE gene MSE')
    print('  - Pearson correlation')
    print('  - Direction error rate')
    print('  - Precision@10')
    print('  - Genetic interaction R^2')
    print('  - ARI / NMI')

## 评估指标说明

| 指标 | 说明 |
|------|------|
| Top-20 DE MSE | 预测与真实 top-20 差异基因的均方误差 |
| Pearson | 预测与真实表达变化的 Pearson 相关系数 |
| Direction error | 预测方向与真实方向不一致的比例 |
| Precision@10 | top-10 预测 DE 基因中的准确率 |
| GI R^2 | 遗传相互作用 (组合效应) 的拟合优度 |
| ARI | 聚类调整兰德指数 |
| NMI | 归一化互信息 |
| Uncertainty | 模型预测的不确定性/方差 |